## Dog Breed Identification 
#### By Sawyer Selby

### Installs

In [ ]:
## Uncomment the below line if you do not have the imports installed on your device locally:
## %pip install kaggle keras scikit-learn pandas matplotlib pillow
## Or in the terminal you can do: pip install kaggle keras scikit-learn pandas matplotlib pillow
## To run with CUDA uncomment below:
## %pip install torch --index-url https://download.pytorch.org/whl/cu128
## Or again in the terminal: pip install torch --index-url https://download.pytorch.org/whl/cu128

### Imports

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch" ## Run Keras on PyTorch. Must be set before importing Keras
import gc
import zipfile
from pathlib import Path

import numpy as np ## Arrays
import pandas as pd ## Tables
import matplotlib.pyplot as plt ## Plots
from PIL import Image ## Opening images
import torch ## Used to free GPU memory between models

import keras
from keras import layers

from sklearn.model_selection import train_test_split ## Train/validation/test split
from sklearn.preprocessing import StandardScaler ## Scaling for the baseline
from sklearn.metrics import confusion_matrix ## Error analysis

### Setup

In [ ]:
DATA_DIR = "data" ## Folder for Kaggle data
COMPETITION = "dog-breed-identification"

tokenPath = Path.home() / ".kaggle" / "access_token"
os.environ["KAGGLE_API_TOKEN"] = tokenPath.read_text().strip()

from kaggle.api.kaggle_api_extended import KaggleApi

os.makedirs(DATA_DIR, exist_ok=True) ## Creates data folder

api = KaggleApi()
api.authenticate() ## Needs an API token to run. Place it in .kaggle/access_token in your home folder for it to authenticate and run

## Downloads the competion zip, skips if it is already there
zipPath = os.path.join(DATA_DIR, f"{COMPETITION}.zip")
if not os.path.exists(zipPath):
    api.competition_download_files(COMPETITION, path=DATA_DIR)

with zipfile.ZipFile(zipPath, "r") as z: ## Unzips the data, creats train, test, labesl.csv, sample_submission.csv
    z.extractall(DATA_DIR)

### Seed and Device

In [ ]:
SEED = 42
keras.utils.set_random_seed(SEED) ## Same random numbers every run

### Labels

In [ ]:
labels = pd.read_csv(os.path.join(DATA_DIR, "labels.csv")) ## Image ID and breed for every training image

counts = labels["breed"].value_counts()
print(f"{labels['breed'].nunique()} breeds, {counts.min()} to {counts.max()} images per breed")

### Visual of the dataset

In [ ]:
## A few example images with their breeds
sample = labels.sample(8, random_state=SEED)

plt.figure(figsize=(16, 8))
for k, (imgId, breed) in enumerate(zip(sample["id"], sample["breed"])):
    plt.subplot(2, 4, k + 1)
    plt.imshow(Image.open(os.path.join(DATA_DIR, "train", f"{imgId}.jpg")))
    plt.title(breed, fontsize=10)
    plt.axis("off")
plt.tight_layout(); plt.show()

### Split

In [ ]:
# 20% test
trainValSplit, testSplit = train_test_split(
    labels,
    test_size=0.2,
    stratify=labels["breed"], ## Same breed mix in every split
    random_state=SEED,
)
## The rest is split into 60% training and 20% validation
trainSplit, valSplit = train_test_split(
    trainValSplit,
    test_size=0.25,
    stratify=trainValSplit["breed"],
    random_state=SEED,
)

print(f"Train: {len(trainSplit)}, Validation: {len(valSplit)}, Test: {len(testSplit)}")

## Check that no image is in more than one split
shared = (set(trainSplit["id"]) & set(valSplit["id"])) | (set(trainSplit["id"]) & set(testSplit["id"])) | (set(valSplit["id"]) & set(testSplit["id"]))
print("Images in more than one split:", len(shared))

## Save the splits
trainSplit.to_csv(os.path.join(DATA_DIR, "train_split.csv"), index=False)
valSplit.to_csv(os.path.join(DATA_DIR, "val_split.csv"), index=False)
testSplit.to_csv(os.path.join(DATA_DIR, "test_split.csv"), index=False)

### Load and Flatten

In [ ]:
IMG_SIZE = 32 ## Small images for the baseline

## Give each breed a number
breeds = sorted(labels["breed"].unique())
breedToIdx = {b: i for i, b in enumerate(breeds)}
NUM_CLASSES = len(breeds)

def load_flat(df):
    X = np.zeros((len(df), IMG_SIZE * IMG_SIZE * 3), dtype=np.float32)
    for i, imgId in enumerate(df["id"]):
        img = Image.open(os.path.join(DATA_DIR, "train", f"{imgId}.jpg")).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE))
        X[i] = np.asarray(img, dtype=np.float32).reshape(-1) / 255.0 ## Flatten into one row of pixel values
    y = df["breed"].map(breedToIdx).to_numpy(dtype=np.int64) ## Breed names to numbers
    return X, y

xTrain, yTrain = load_flat(trainSplit)
xVal, yVal = load_flat(valSplit)
xTest, yTest = load_flat(testSplit)

## Scale using the training images only
scaler = StandardScaler()
xTrain = scaler.fit_transform(xTrain)
xVal = scaler.transform(xVal)
xTest = scaler.transform(xTest)

### Train Baseline

In [ ]:
baseline = keras.Sequential([
    keras.Input(shape=(IMG_SIZE * IMG_SIZE * 3,)), ## One input per pixel
    layers.Dense(NUM_CLASSES, activation="softmax"), ## One output per breed. Softmax turns the scores into probabilities
])

baseline.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01), ## Plain SGD. Learning rate of 0.01
    loss="sparse_categorical_crossentropy", ## Cross-entropy equals log loss. "sparse" means the labels are numbers
    metrics=["accuracy"],
)

historyBaseline = baseline.fit(
    xTrain, yTrain,
    validation_data=(xVal, yVal), ## Scored on the validation set after every epoch
    epochs=30, ## Number of passes over the training set
    batch_size=128, ## Number of images per gradient update. One minibatch
    verbose=2, ## One line per epoch
)

### Plot learning curves

In [ ]:
def plot_curves(history, title):
    hist = history.history
    epochs = range(1, len(hist["loss"]) + 1)
    bestEpoch = int(np.argmin(hist["val_loss"])) + 1 ## Epoch with the lowest validation loss

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ## Loss
    ax1.plot(epochs, hist["loss"], label="Training")
    ax1.plot(epochs, hist["val_loss"], label="Validation")
    ax1.axvline(bestEpoch, linestyle="--", color="green", label=f"Best epoch ({bestEpoch})")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Log loss")
    ax1.set_title(f"{title}: Loss"); ax1.legend()

    ## Accuracy
    ax2.plot(epochs, hist["accuracy"], label="Training")
    ax2.plot(epochs, hist["val_accuracy"], label="Validation")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.set_title(f"{title}: Accuracy"); ax2.legend()

    plt.tight_layout()
    plt.show()

plot_curves(historyBaseline, "Baseline")

### Baseline Test Score

In [ ]:
results = [] ## One row per model

def record(name, history, model, xTestData, yTestData):
    testLoss, testAcc = model.evaluate(xTestData, yTestData, verbose=0) ## Score on the test set
    hist = history.history
    results.append({
        "Model": name,
        "Best val loss": min(hist["val_loss"]),
        "Best val acc": max(hist["val_accuracy"]),
        "Train acc": hist["accuracy"][-1],
        "Test loss": testLoss,
        "Test acc": testAcc,
    })

record("Baseline", historyBaseline, baseline, xTest, yTest)

### Load images at full size

In [ ]:
RESNET_SIZE = 224 ## Image size ResNet-50 was trained on

def load_images(df):
    X = np.zeros((len(df), RESNET_SIZE, RESNET_SIZE, 3), dtype=np.uint8) ## Whole numbers to save memory
    for i, imgId in enumerate(df["id"]):
        img = Image.open(os.path.join(DATA_DIR, "train", f"{imgId}.jpg")).convert("RGB")
        X[i] = np.asarray(img.resize((RESNET_SIZE, RESNET_SIZE))) ## Keep the image shape. No flattening
    return X

imgTrain = load_images(trainSplit)
imgVal = load_images(valSplit)
imgTest = load_images(testSplit)

### Build the frozen model

In [ ]:
base = keras.applications.ResNet50(
    weights="imagenet", ## Pretrained on ImageNet
    include_top=False, ## Remove the ImageNet output layer
    input_shape=(RESNET_SIZE, RESNET_SIZE, 3),
)
base.trainable = False ## Freeze all ResNet-50 layers

inputs = keras.Input(shape=(RESNET_SIZE, RESNET_SIZE, 3))
x = layers.Lambda(keras.applications.resnet50.preprocess_input)(inputs) ## Prepare pixels the way ResNet-50 expects
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x) ## New output layer for the 120 breeds

frozenModel = keras.Model(inputs, outputs)
frozenModel.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
frozenModel.summary() ## Shows trainable vs. frozen weights

### Train the frozen model

In [ ]:
historyFrozen = frozenModel.fit(
    imgTrain, yTrain,
    validation_data=(imgVal, yVal),
    epochs=15,
    batch_size=64,
    verbose=2,
)

plot_curves(historyFrozen, "Frozen")
record("Frozen", historyFrozen, frozenModel, imgTest, yTest)

### Finetune the last block

In [ ]:
base.trainable = True ## Make ResNet-50 trainable again.
for layer in base.layers:
    layer.trainable = layer.name.startswith("conv5_") ## Refreeze everything except the last block

print("Trainable ResNet-50 layers:", sum(layer.trainable for layer in base.layers), "of", len(base.layers))

### Train with a small learning rate

In [ ]:
## Recompile so the change takes effect
frozenModel.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5), ## Small steps to keep the pretrained weights
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

historyFineTuned = frozenModel.fit(
    imgTrain, yTrain,
    validation_data=(imgVal, yVal),
    epochs=10,
    batch_size=64,
    verbose=2,
)

plot_curves(historyFineTuned, "Fine-tuned conv5")
record("Fine-tuned conv5", historyFineTuned, frozenModel, imgTest, yTest)

### Experiment function

In [ ]:
## Builds and trains a fresh frozen ResNet-50 with one change at a time
def build_and_train(augment=False, dropout=0.0, earlyStop=False):
    ## Free GPU memory from the last model
    keras.backend.clear_session()
    gc.collect()
    torch.cuda.empty_cache()

    keras.utils.set_random_seed(SEED) ## Same start for every experiment

    base = keras.applications.ResNet50(weights="imagenet", include_top=False,
                                       input_shape=(RESNET_SIZE, RESNET_SIZE, 3))
    base.trainable = False

    inputs = keras.Input(shape=(RESNET_SIZE, RESNET_SIZE, 3))
    x = inputs
    if augment:
        x = layers.RandomFlip("horizontal")(x) ## Randomly mirror training images
    x = layers.Lambda(keras.applications.resnet50.preprocess_input)(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x) ## Randomly turn off features while training
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = keras.Model(inputs, outputs)

    callbacks = []
    if earlyStop: ## Stop when validation loss stops improving and keep the best epoch
        callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history = model.fit(imgTrain, yTrain, validation_data=(imgVal, yVal),
                        epochs=15, batch_size=64, callbacks=callbacks, verbose=2)
    return model, history

## Train, plot, and record one experiment
def run(name, **changes):
    model, history = build_and_train(**changes)
    plot_curves(history, name)
    record(name, history, model, imgTest, yTest)
    return model

### Experiments

In [ ]:
_ = run("Frozen (rerun)")

In [ ]:
_ = run("Early stopping", earlyStop=True)

In [ ]:
_ = run("Dropout", dropout=0.5)

In [ ]:
_ = run("Flip", augment=True)

In [ ]:
finalModel = run("Flip plus Early Stopping", augment=True, earlyStop=True)

### All results

In [ ]:
display(pd.DataFrame(results).round(4))

### Error analysis

In [ ]:
probs = finalModel.predict(imgTest, batch_size=64, verbose=0) ## Probability for each breed
preds = probs.argmax(axis=1) ## Most likely breed
cm = confusion_matrix(yTest, preds, labels=np.arange(NUM_CLASSES))

plt.figure(figsize=(9, 8))
plt.imshow(cm, cmap="Blues")
plt.colorbar(label="Images")
plt.xlabel("Predicted breed"); plt.ylabel("True breed")
plt.title("Confusion Matrix (Test Set)")
plt.show()

### Most confused breeds and weakest breeds

In [ ]:
mistakes = cm.copy()
np.fill_diagonal(mistakes, 0) ## Keep only wrong predictions

## 10 most common mistakes
topIdx = np.argsort(mistakes, axis=None)[::-1][:10]
rows, cols = np.unravel_index(topIdx, mistakes.shape)
confusedPairs = pd.DataFrame({
    "True": [breeds[r] for r in rows],
    "Predicted": [breeds[c] for c in cols],
    "Count": [mistakes[r, c] for r, c in zip(rows, cols)],
})
display(confusedPairs)

## 10 breeds with the lowest accuracy
breedAcc = cm.diagonal() / cm.sum(axis=1)
worstBreeds = pd.DataFrame({"Breed": breeds, "Accuracy": breedAcc}).sort_values("Accuracy").head(10)

### Most confident mistakes

In [ ]:
wrong = np.where(preds != yTest)[0] ## Test images the model got wrong
wrongConf = probs[wrong, preds[wrong]] ## How confident it was in the wrong answer
show = wrong[np.argsort(wrongConf)[::-1][:8]] ## The 8 most confident mistakes

plt.figure(figsize=(16, 8))
for k, i in enumerate(show):
    plt.subplot(2, 4, k + 1)
    plt.imshow(imgTest[i])
    plt.title(f"True: {breeds[yTest[i]]}\nPredicted: {breeds[preds[i]]} ({probs[i, preds[i]]:.0%})", fontsize=9)
    plt.axis("off")
plt.tight_layout(); plt.show()

### Comparison with the published results

In [ ]:
baseRow, finalRow = results[0], results[-1]

comparison = pd.DataFrame([
    {"Model": "Baseline (ours)", "Test set": "Our split", "Accuracy": baseRow["Test acc"], "Log loss": baseRow["Test loss"]},
    {"Model": "ResNet-50 final (ours)", "Test set": "Our split", "Accuracy": finalRow["Test acc"], "Log loss": finalRow["Test loss"]},
    {"Model": "GoogLeNet (Sermanet et al., 2014)", "Test set": "Stanford Dogs", "Accuracy": 0.755, "Log loss": None},
    {"Model": "ResNet-50 (Nguyen et al., PCNN)", "Test set": "Stanford Dogs", "Accuracy": 0.8582, "Log loss": None},
    {"Model": "Pretrained CNNs (Vadla, Kaggle)", "Test set": "Kaggle leaderboard", "Accuracy": None, "Log loss": 0.2589},
])
display(comparison.round(4))